# Example: Net Present Value for a Tesla Model S
In this example, we represent the purchase, ownership, and [sale of a Tesla Model S](https://www.tesla.com/models) as dated signed cash flows and compute their time-0 net present value.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct dated net cash flows:__ Represent the purchase price, the recurring ownership costs, and the terminal sale value as a single signed net cash flow at each period of the ownership horizon. The sign convention carries the direction of each payment, so an inflow and an outflow at the same date combine into one number.
> * __Compute accumulation and discount factors:__ Build the forward accumulation factor for every period from a stated nominal annual yield and compounding frequency, then invert it to obtain the time-0 discount factors. Because these factors depend only on the rate convention, the same array discounts any cash-flow stream defined on the same period grid.
> * __Calculate and interpret NPV:__ Evaluate the net present value as the scalar product of the discount-factor and net-cash-flow arrays, and read its sign against the benchmark that was declared. Decomposing the result into its purchase, ownership, and resale components shows which term drives the sign.

Let's construct the ownership cash flows and value them at time $0$. Is this a good investment?
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, activates the course project, and loads the required external packages.

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # activate the course project and load the shared L1b packages

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Problem Components and Constants
Before computing NPV, we specify the valuation convention, horizon, and cash-flow assumptions. We use the package's [`DiscreteCompoundingModel`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) to generate forward accumulation factors.

> __Notation:__ These symbols carry over from [the lecture](./CHEME-5660-L1b-Lecture-TimeValueMoney-Fall-2026.ipynb).
>
> * $y$ is the nominal annual yield and $n$ is the number of compounding periods per year, so $y/n$ is the rate per period.
> * $T$ is the ownership horizon in years, $j=0,1,\ldots,N$ indexes compounding periods from the valuation date, and $N=nT$ is the final index.
> * $\mathcal D_{j,0}(y;n)=(1+y/n)^{j}$ is the forward accumulation factor from period $0$ to period $j$, and its inverse $\mathcal D^{-1}_{j,0}(y;n)$ is the time-0 discount factor for period $j$.
> * $\bar c_j$ is the signed net cash flow at index $j$, positive for an inflow and negative for an outflow. The vectors $\bar{\mathbf c}$ and $\mathcal D^{-1}_{\star,0}(y;n)$ are the ordered $(N+1)$-tuples whose entries run over $j=0,1,\ldots,N$.

The package method is named `discount(...)` for compatibility, although the values it returns are forward accumulation factors: for $y>0$ they satisfy $\mathcal D_{j,0}(y;n)\geq 1$, with equality at $j=0$. Their inverses are the time-0 discount factors used in the NPV calculation. The cell below stores the model in `compounding_model::DiscreteCompoundingModel`.

In [2]:
compounding_model = DiscreteCompoundingModel(); # model used to generate period-indexed accumulation factors

Next, assign values to the horizon $T$, the compounding frequency $n$, the nominal annual yield $y$, and the per-period depreciation rate $\delta$. The cell below stores `T::Float64`, `n::Int64`, `y::Float64`, `depreciation::Float64`, and the final cash-flow index `N::Int64`, and guards the modeling assumptions with [`@assert`](https://docs.julialang.org/en/v1/base/base/#Base.@assert).

In [3]:
# Initialize the time grid and rates used by the cash-flow and valuation models -
T = 10.0;   # ownership horizon (years)
n = 2;      # compounding periods per year; n = 2 gives semiannual periods
y = 0.0425; # nominal annual yield used for valuation (year⁻¹)
depreciation = 0.0625; # fraction of the remaining vehicle value lost per period

# Check the model domain before using these values in powers or array sizes -
@assert isinteger(n*T) "T = $(T) yr at n = $(n) per yr does not land on the compounding grid";
@assert 0 ≤ depreciation < 1 "depreciation must satisfy 0 ≤ depreciation < 1";
@assert 1 + y/n > 0 "1 + y/n must be positive; got y = $(y) at n = $(n)";

N = round(Int, n*T); # integer index of the final semiannual period

___

## Task 1: Construct the Net Cash-Flow Dictionary
Specify the cash-flow events over the Tesla Model S ownership horizon, then store the signed net cash flow $\bar c_j$ at each index $j=0,\ldots,N$.

Three assumptions set the magnitudes. We pay the purchase price $P$ at $j=0$. We pay insurance and other ownership costs at the end of every period $j=1,\ldots,N$ and credit any savings over the same periods. We sell the car at $j=N$ for a resale value $S$ set by a __declining-balance__ model, in which the car loses a fixed fraction $\delta$ of its _remaining_ value each period:
$$
S=P\left(1-\delta\right)^{N}.
$$
At $\delta=6.25\%$ per semiannual period, the car retains $(1-\delta)^{2}=87.9\%$ of its value each year and $27.5\%$ of the purchase price after ten years. These are illustrative figures rather than market quotes, so substitute your own.

The cell below stores `purchase_price::Int64`, the derived `sale_price::Float64`, and the recurring `insurance_costs::Float64`, `other_costs::Float64`, and `other_savings::Float64`.

In [4]:
# Initialize cash-flow magnitudes in USD; signs are assigned when the schedule is built -
purchase_price = 111630;  # time-0 purchase price
insurance_costs = 1808.0; # insurance paid at the end of each semiannual period
other_costs = 50.0;       # other costs paid at the end of each semiannual period
other_savings = 0.0;      # savings credited at the end of each semiannual period
sale_price = purchase_price*(1 - depreciation)^N; # terminal value under declining-balance depreciation, P(1-δ)^N

The `cash_flow_event_dictionary::Dict{Int64,Float64}` variable maps each cash-flow index $j=0,\ldots,N$ to its signed net cash flow $\bar c_j$.

* At $j=0$, the purchase price is an outflow.
* For $0<j<N$, recurring insurance and other costs are netted against any savings.
* At $j=N$, the sale value and final-period savings are netted against the final-period costs.

Each case above is the inner product $\bar c_j=\left\langle\mathbf c_j,\boldsymbol\nu_j\right\rangle$ introduced in the lecture, where $\mathbf c_j\in\mathbb R^{m_j}_{\geq0}$ collects the $m_j$ nonnegative cash-flow magnitudes at index $j$ and $\boldsymbol\nu_j\in\{-1,+1\}^{m_j}$ carries their directions. The purchase date has $m_0=1$, with $\mathbf c_0=(\texttt{purchase\_price})$ and $\boldsymbol\nu_0=(-1)$. A recurring period $0<j<N$ has $m_j=3$, with $\mathbf c_j=(\texttt{other\_savings},\ \texttt{insurance\_costs},\ \texttt{other\_costs})$ and $\boldsymbol\nu_j=(+1,-1,-1)$. The terminal period adds the sale value with direction $+1$, so $m_N=4$.

We populate the dictionary by iterating over the cash-flow indices with a Julia [`for` loop](https://docs.julialang.org/en/v1/base/base/#for).

In [5]:
cash_flow_event_dictionary = let

    # Initialize -
    cash_flow_event_dictionary = Dict{Int64,Float64}(); # map each financial index j to one signed net cash flow

    # Populate the schedule using positive inflows and negative outflows -
    for j ∈ 0:N
        if j == 0
            cash_flow_event_dictionary[j] = -purchase_price; # immediate purchase outflow
        elseif j == N
            cash_flow_event_dictionary[j] = sale_price + other_savings - (insurance_costs + other_costs); # resale plus final recurring cash flow
        else
            cash_flow_event_dictionary[j] = other_savings - (insurance_costs + other_costs); # intermediate recurring net cash flow
        end
    end

    cash_flow_event_dictionary # return the populated dictionary from the `let` block
end;

___

## Task 2: Compute the Accumulation-Factor Dictionary
Compute $\mathcal D_{j,0}(y;n)$ for $j=0,\ldots,N$ using the stated nominal annual yield and compounding frequency. We call the package's [`discount(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.discount-Tuple%7BAbstractCompoundingModel,%20Float64,%20Int64%7D), which retains its legacy name but returns the forward factors in a dictionary.

The function takes the compounding frequency through the keyword `λ`, so we pass `λ = n`. The cell below stores the result in `accumulation_dictionary::Dict{Int64,Float64}`, keyed by the period index $j$.

In [6]:
accumulation_dictionary = discount(compounding_model, y, N, λ = n); # legacy API returns forward factors 𝒟(j,0)

`Unhide` the code block below to see the factors for every period. The `accumulation` column reports $\mathcal D_{j,0}(y;n)$, and the `discount` column reports its inverse. The table is constructed with [`pretty_table(...)`](https://github.com/ronisbr/PrettyTables.jl) and a [`DataFrame`](https://github.com/JuliaData/DataFrames.jl).

In [7]:
let
    # Initialize -
    df = DataFrame(); # reader-facing table; the source dictionary remains unchanged

    # Populate one chronologically ordered row per period -
    for j ∈ 0:N
        value = accumulation_dictionary[j]; # select the factor by financial index
        row_df = (
            period = j,
            accumulation = value, # forward factor from time 0 to period j
            discount = 1/value,   # inverse factor that returns period-j value to time 0
        );
        push!(df, row_df); # append the completed row to the table
    end

    # Display all periods without vertical cropping -
    pretty_table(df;
        table_format = TextTableFormat(borders = text_table_borders__simple),
        fit_table_in_display_vertically = false)
end;

========= ============== ===========
  period   accumulation   discount
   Int64        Float64    Float64
========= ============== ===========
       0            1.0        1.0
       1        1.02125   0.979192
       2        1.04295   0.958817
       3        1.06511   0.938866
       4        1.08775   0.919331
       5        1.11086   0.900201
       6        1.13447    0.88147
       7        1.15858   0.863129
       8         1.1832   0.845169
       9        1.20834   0.827583
      10        1.23402   0.810362
      11        1.26024   0.793501
      12        1.28702    0.77699
      13        1.31437   0.760822
      14         1.3423   0.744991
      15        1.37082   0.729489
      16        1.39995    0.71431
      17         1.4297   0.699447
      18        1.46008   0.684893
      19        1.49111   0.670642
      20        1.52279   0.656687
========= ============== ===========


### Check: Do We Recover the Nominal Annual Yield $y$?
For $j\geq1$, invert
$$
\mathcal D_{j,0}(y;n)=\left(1+\frac{y}{n}\right)^j
$$
to obtain
$$
\boxed{
y=n\left(\mathcal D_{j,0}^{1/j}-1\right).
}
$$

Iterate over the accumulation-factor dictionary, recover $y$ at each positive index, and compare it with the specified value using Julia's [`@assert`](https://docs.julialang.org/en/v1/base/base/#Base.@assert) and [`isapprox(...)`](https://docs.julialang.org/en/v1/base/math/#Base.isapprox).

In [8]:
let
    # Check the recovered yield at every financial index -
    for (j, 𝒟ⱼ) ∈ accumulation_dictionary
        if j == 0 # the zeroth-power factor contains no rate information
            @assert isapprox(𝒟ⱼ, 1.0) "𝒟(0,0) must equal 1, got $(𝒟ⱼ)";
            continue; # skip the inversion at j = 0
        end

        yⱼ = n*(𝒟ⱼ^(1/j) - 1); # invert 𝒟(j,0) = (1 + y/n)^j
        @assert isapprox(y, yⱼ, rtol = 1e-4) "recovered y = $(yⱼ) at j = $(j), expected $(y)";
    end
end;

___

## Task 3: Compute and Interpret Net Present Value
Combine the signed net-cash-flow schedule $\bar c_j$ with the time-0 discount factors $\mathcal D_{j,0}^{-1}(y;n)$. Multiplying each cash flow by the discount factor for the same period converts it to time-0 dollars. Adding the discounted cash flows gives
$$
\boxed{
\operatorname{NPV}_0(y)
=\sum_{j=0}^{N}\mathcal D_{j,0}^{-1}(y;n)\bar c_j
=\left\langle\mathcal D_{\star,0}^{-1}(y;n),\bar{\mathbf c}\right\rangle.
}
$$

The scalar product performs two operations: it discounts each dated cash flow and sums the resulting present values. To pair each cash flow with the correct discount factor, construct both arrays in chronological order, $j=0,\ldots,N$, and evaluate their scalar product with [`dot(...)`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot).

The resulting NPV measures the value of the specified cash-flow schedule relative to the yield $y$. It does not include the value of using the car unless that benefit is represented explicitly as a cash flow.

### Construct the Discount-Factor Array $\mathcal D^{-1}_{\star,0}(y;n)$
Julia arrays use one-based positions, while the financial indices run from $j=0$ through $N$. At array position $i$, set $j=i-1$ and store the inverse of `accumulation_dictionary[j]`. The cell below stores the result in `𝒟inv::Vector{Float64}`, where entry $i$ holds $\mathcal D^{-1}_{i-1,0}(y;n)$.

In [9]:
𝒟inv = let
    # Initialize -
    𝒟inv = Array{Float64,1}(undef, N+1); # preallocate one factor for each financial index j = 0,...,N

    for i ∈ 1:(N+1)
        j = i - 1; # Julia position i corresponds to financial index j
        𝒟inv[i] = 1/accumulation_dictionary[j]; # invert the aligned accumulation factor
    end

    𝒟inv # return the populated vector from the `let` block
end;

### Construct the Net-Cash-Flow Array $\bar{\mathbf c}$
At array position $i$, set $j=i-1$ and copy $\bar c_j$ from the cash-flow dictionary into `c̄`. The cell below stores the result in `c̄::Vector{Float64}`.

In [10]:
c̄ = let
    # Initialize -
    c̄ = Array{Float64,1}(undef, N+1); # preallocate one cash flow for each financial index j = 0,...,N

    for i ∈ 1:(N+1)
        j = i - 1; # Julia position i corresponds to financial index j
        c̄[i] = cash_flow_event_dictionary[j]; # copy the aligned signed cash flow
    end

    c̄ # return the populated vector from the `let` block
end;

### Inspect the Period-by-Period Valuation
Construct one row for each financial index $j=0,\ldots,N$. The `cbar_USD` column reports $\bar c_j$ without a timing adjustment, the `discount` column reports $\mathcal D_{j,0}^{-1}(y;n)$, and the `PV0_USD` column reports their product. The total of the time-0 values is $\operatorname{NPV}_0(y)$.

The nominal total adds cash flows that occur at different dates, so it is not a time-0 valuation. It is included only to show how discounting changes the cash-flow schedule.

In [11]:
NPV = let
    # Compute the net present value -
    npv = dot(𝒟inv, c̄); # multiply aligned cash flows and discount factors, then sum

    # Label the period rows for the audit table -
    events = [j == 0 ? "purchase" :
              j == N ? "recurring + resale" :
                       "recurring" for j ∈ 0:N]; # one event label for each financial index

    # Assemble the period rows and a final total row -
    df = DataFrame(
        j = [string.(0:N); "Total"], # strings allow the last row to be labeled `Total`
        event = [events; ""],        # the total row has no single cash-flow event
        cbar_USD = [c̄; sum(c̄)], # nominal cash flows, without a timing adjustment
        discount = Union{Missing,Float64}[𝒟inv...; missing], # no single factor applies to the total row
        PV0_USD = [𝒟inv .* c̄; npv], # discounted contribution of each row
    );

    # Display cash values to cents and discount factors to six decimal places -
    pretty_table(df;
        table_format = TextTableFormat(borders = text_table_borders__simple),
        fit_table_in_display_vertically = false,
        formatters = [fmt__printf("%.2f", [3, 5]), fmt__printf("%.6f", [4])]);

    println("The time-0 NPV for a Tesla Model S over $(Int(T)) years is $(round(npv, digits = 2)) USD."); # report the scalar result

    npv # return the scalar so the outer assignment stores it as `NPV`
end;

========= ==================== ============ ========== =============
       j                event     cbar_USD   discount      PV0_USD 
  String               String      Float64   Float64?      Float64 
========= ==================== ============ ========== =============
       0             purchase   -111630.00   1.000000   -111630.00
       1            recurring     -1858.00   0.979192     -1819.34
       2            recurring     -1858.00   0.958817     -1781.48
       3            recurring     -1858.00   0.938866     -1744.41
       4            recurring     -1858.00   0.919331     -1708.12
       5            recurring     -1858.00   0.900201     -1672.57
       6            recurring     -1858.00   0.881470     -1637.77
       7            recurring     -1858.00   0.863129     -1603.69
       8            recurring     -1858.00   0.845169     -1570.32
       9            recurring     -1858.00   0.827583     -1537.65
      10            recurring     -1858.00   0.810362   

### Reading the Result
Split the NPV into the three parts of the schedule, all measured in time-0 dollars, to see which term dominates.

In [12]:
let
    # Initialize -
    recurring = other_savings - (insurance_costs + other_costs); # signed recurring cash flow per period

    # Compute each economic component in time-0 dollars -
    pv_purchase = -purchase_price;                                       # purchase is already at j = 0
    pv_costs    = sum(recurring/accumulation_dictionary[j] for j ∈ 1:N); # recurring stream at j = 1,...,N
    pv_resale   = sale_price/accumulation_dictionary[N];                 # terminal resale receipt at j = N

    # Assemble the component summary -
    df = DataFrame(
        component = ["purchase at j = 0", "ownership costs", "resale at j = N", "NPV"],
        time_0_USD = [pv_purchase, pv_costs, pv_resale,
                      pv_purchase + pv_costs + pv_resale], # recombine the components to recover NPV
    );

    # Display every component in USD rounded to cents -
    pretty_table(df;
        table_format = TextTableFormat(borders = text_table_borders__simple),
        formatters = [fmt__printf("%.2f", [2])])
end;

==================== =============
          component   time_0_USD 
             String      Float64 
==================== =============
  purchase at j = 0   -111630.00
    ownership costs    -30017.65
    resale at j = N     20163.46
                NPV   -121484.19
==================== =============


The purchase price is the largest single term, and discounting shrinks the resale receipt to about two-thirds of its nominal value because it arrives $N$ periods after the valuation date.

A negative NPV here does not mean that buying the car is a mistake. The model prices only the cash flows we listed, and it assigns no value to using the car over the horizon. What it reports is the time-0 cost of ten years of ownership, on cash flows alone and under the declared benchmark. Comparing that figure against what the same dollars would buy elsewhere is the decision, and NPV is one input to it.

### Discussion Questions
* What does a negative NPV indicate about this investment?
* What factors could we change to improve the NPV of the Tesla Model S?

___

## Summary
We represented ten years of Model S ownership as dated signed cash flows and valued the whole schedule at time zero.

> __Key Takeaways:__
>
> * __An asset is its cash-flow schedule:__ Once the purchase, the recurring costs, and the resale value are written as one signed amount per period, a car is valued with the same machinery as any other asset. Nothing in the calculation depends on what the asset physically is.
> * __Discount factors depend only on the convention:__ The accumulation factors follow from the stated yield and compounding frequency alone, so inverting them once gives a discount-factor array that any cash-flow stream on the same grid can reuse. Recovering the yield from those factors confirms that the convention was applied as stated.
> * __The negative result is what the model measures:__ The purchase price and the discounted ownership costs together exceed the discounted resale value, so NPV is negative under the benchmark we declared. The model prices cash flows only, so the result describes this schedule at this discount rate rather than delivering a verdict on owning a car.

The same three steps (build the cash flows, build the discount factors, then take their scalar product) carry over to every valuation later in the course.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___